# U2T01 — SQuAD runs on a Kaggle GPU

Runs **the same modules as the repository**, on a P100 or T4 instead of the local Mac.
Nothing is reimplemented: `src/` is the single source of truth, so the `results/*.json`
these cells produce drop straight into the report next to the locally-trained runs.

**Before running:** in the right-hand panel set *Accelerator* to **GPU P100** (or T4 x2)
and switch *Internet* **on** — both need a phone-verified account, and without internet
the datasets and the `bert-base-uncased` weights cannot be downloaded.


## 1. Point at the code

Either attach the project zip as a Kaggle Dataset (*+ Add Input → Upload*), or clone it
from GitHub if the repository is public.


In [ ]:
import os, pathlib, shutil, glob

GITHUB_REPO = ''   # e.g. 'https://github.com/joseeangel0/adapting-bert-nlp-tasks.git'
WORK = pathlib.Path('/kaggle/working/u2t01')

if GITHUB_REPO:
    !git clone -q $GITHUB_REPO {WORK}
else:
    # Find the uploaded zip (or already-unzipped folder) among the attached inputs.
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if zips:
        !mkdir -p {WORK} && unzip -q -o "{zips[0]}" -d {WORK}
    else:
        src = next(p for p in pathlib.Path('/kaggle/input').rglob('src') if (p/'train.py').exists())
        shutil.copytree(src.parent, WORK, dirs_exist_ok=True)
    # A zip often contains a single top-level folder; step into it.
    if not (WORK/'src').is_dir():
        WORK = next(p for p in WORK.iterdir() if (p/'src').is_dir())

os.chdir(WORK)
print('working directory:', pathlib.Path.cwd())
print('contents:', sorted(p.name for p in pathlib.Path('.').iterdir())[:12])


## 2. Dependencies

Kaggle images already carry a CUDA build of torch; installing the pinned wheel would
replace it, so torch is filtered out and the rest is installed at the pinned versions.


In [ ]:
!grep -v '^torch==' requirements.txt > /tmp/req_kaggle.txt
!pip install -q -r /tmp/req_kaggle.txt 2>&1 | tail -2

import torch, transformers, datasets
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
print('transformers', transformers.__version__, '| datasets', datasets.__version__)
assert torch.cuda.is_available(), 'Turn the GPU accelerator on in the right-hand panel'


## 3. Check the alignment before training anything

A silent subword/label misalignment still produces a healthy-looking loss curve.


In [ ]:
!python scripts/verify_data.py 2>&1 | grep -E '^(PASS|FAIL)'


## 4. Run SQuAD

The three adaptation methods for extractive QA: a frozen span head, partial fine-tuning of
the top four encoder layers, and full fine-tuning. Resumable — a run whose
`results/qa/<run_id>.json` already exists is skipped, which matters if the session restarts.

Expect roughly 18–25 minutes on a P100 for all three.


In [ ]:
!python -u scripts/run_experiments.py --task qa


### If you want the rest too

Only needed if the local machine has not finished them; already-present results are skipped.


In [ ]:
!python -u scripts/run_experiments.py


## 5. Look at what came out


In [ ]:
import sys; sys.path.insert(0, '.')
from src.report_data import rows
for r in rows('qa'):
    if r['headline'] is not None:
        print(f"{r['method']:34s} F1 {r['headline']:6.2f}  EM {r['secondary']:6.2f}  "
              f"trainable={r['trainable']:>11,}  {r['minutes']:5.1f} min")


## 6. Take the results home

`results/` holds every metric, loss curve and timing — that is what the report reads.
`models/qa/` holds the checkpoints that get published to the Hub.
Both land in `/kaggle/working`, downloadable from the *Output* panel on the right.


In [ ]:
!cd /kaggle/working/u2t01 2>/dev/null || cd {WORK}; zip -qr /kaggle/working/u2t01_qa_results.zip results docs
!du -sh /kaggle/working/u2t01_qa_results.zip
print('Also download models/qa/ from the Output panel if you will publish from here.')
